# Credit Risk XAI & Fairness Audit — Portfolio Walkthrough
This compact notebook exposes the main analytical path preserved from the original audit notebook: synthetic data → model → performance → fairness → threshold sensitivity → governance.


In [ ]:
from src.data_generation import make_credit_data
from src.fairness_metrics import group_metrics, disparity_summary, bootstrap_selection_ratio
from src.threshold_analysis import threshold_sweep
from src.governance import deployment_recommendation
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [ ]:
df = make_credit_data()
X = df.drop(columns=['default', 'audit_group'])
y = df['default']
g = df['audit_group']
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(X, y, g, test_size=0.30, stratify=y, random_state=42)
cat = X.select_dtypes(exclude='number').columns.tolist()
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
model = Pipeline([('pre', pre), ('rf', RandomForestClassifier(n_estimators=350, min_samples_leaf=8, random_state=42))])
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, prob)
auc


In [ ]:
metrics = group_metrics(y_test, prob, g_test, threshold=0.50)
summary = disparity_summary(metrics)
ci = bootstrap_selection_ratio(y_test.to_numpy(), prob, g_test.to_numpy(), threshold=0.50)
metrics, summary, ci


In [ ]:
sweep = threshold_sweep(y_test, prob, g_test)
sweep.head(), deployment_recommendation(summary['selection_ratio_min_max'], summary['favorable_opportunity_gap'], auc)
